In [1]:

!pip install -q -U langgraph


In [2]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Step 1: Define the information our application uses
class StudyState(TypedDict):
    student_question: str
    answer: str

# Step 2: Create a node
def study_assistant(state: StudyState):
    question = state["student_question"]

    return {
        "answer": f"Welcome to your AI Study Assistant! You asked: {question}"
    }

# Step 3: Create the graph
builder = StateGraph(StudyState)

# Step 4: Add our node
builder.add_node("study_assistant", study_assistant)

# Step 5: Connect the workflow
builder.add_edge(START, "study_assistant")
builder.add_edge("study_assistant", END)

# Step 6: Compile the graph
graph = builder.compile()

# Step 7: Test the workflow
result = graph.invoke({
    "student_question": "Explain Artificial Intelligence",
    "answer": ""
})

print(result["answer"])


Welcome to your AI Study Assistant! You asked: Explain Artificial Intelligence


In [3]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Information passed through the graph
class StudyState(TypedDict):
    student_question: str
    task: str
    answer: str

# Node 1: Identify the student's task
def classify_task(state: StudyState):
    question = state["student_question"].lower()

    if "quiz" in question or "question" in question:
        task = "quiz"
    elif "plan" in question or "schedule" in question:
        task = "plan"
    else:
        task = "explain"

    return {"task": task}

# Node 2: Explain a topic
def explain_topic(state: StudyState):
    return {"answer": "The assistant will explain your topic here."}

# Node 3: Prepare a study plan
def make_plan(state: StudyState):
    return {"answer": "The assistant will prepare your study plan here."}

# Node 4: Prepare a quiz
def make_quiz(state: StudyState):
    return {"answer": "The assistant will prepare your quiz here."}

# Choose the correct next node
def choose_task(state: StudyState):
    return state["task"]

# Build the graph
builder = StateGraph(StudyState)

builder.add_node("classify", classify_task)
builder.add_node("explain", explain_topic)
builder.add_node("plan", make_plan)
builder.add_node("quiz", make_quiz)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    choose_task,
    {
        "explain": "explain",
        "plan": "plan",
        "quiz": "quiz"
    }
)

builder.add_edge("explain", END)
builder.add_edge("plan", END)
builder.add_edge("quiz", END)

graph = builder.compile()

# Test the workflow
result = graph.invoke({
    "student_question": "Make a study plan for Python",
    "task": "",
    "answer": ""
})

print("Selected task:", result["task"])
print("Result:", result["answer"])


Selected task: plan
Result: The assistant will prepare your study plan here.


In [4]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class StudyState(TypedDict):
    student_question: str
    task: str
    answer: str

# Identify what the student wants
def classify_task(state: StudyState):
    question = state["student_question"].lower()

    if "quiz" in question or "question" in question:
        task = "quiz"
    elif "plan" in question or "schedule" in question:
        task = "plan"
    else:
        task = "explain"

    return {"task": task}

# Give a basic topic explanation
def explain_topic(state: StudyState):
    topic = state["student_question"]

    answer = (
        f"Topic: {topic}\n\n"
        "1. Understand the basic meaning of the topic.\n"
        "2. Learn its important concepts.\n"
        "3. Study simple examples.\n"
        "4. Revise the key points."
    )
    return {"answer": answer}

# Create a simple study plan
def make_plan(state: StudyState):
    answer = (
        "Your Study Plan\n\n"
        "Day 1: Learn the basic concepts.\n"
        "Day 2: Study important topics.\n"
        "Day 3: Practise examples.\n"
        "Day 4: Solve practice questions.\n"
        "Day 5: Revise everything."
    )
    return {"answer": answer}

# Create a sample quiz
def make_quiz(state: StudyState):
    answer = (
        "Sample Quiz\n\n"
        "1. What is the main purpose of studying a topic?\n"
        "2. Why is practice important?\n"
        "3. How can revision improve learning?"
    )
    return {"answer": answer}

def choose_task(state: StudyState):
    return state["task"]

# Build the graph
builder = StateGraph(StudyState)

builder.add_node("classify", classify_task)
builder.add_node("explain", explain_topic)
builder.add_node("plan", make_plan)
builder.add_node("quiz", make_quiz)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    choose_task,
    {
        "explain": "explain",
        "plan": "plan",
        "quiz": "quiz"
    }
)

builder.add_edge("explain", END)
builder.add_edge("plan", END)
builder.add_edge("quiz", END)

graph = builder.compile()

# Test the study planner
result = graph.invoke({
    "student_question": "Make a study plan for Python",
    "task": "",
    "answer": ""
})

print(result["answer"])


Your Study Plan

Day 1: Learn the basic concepts.
Day 2: Study important topics.
Day 3: Practise examples.
Day 4: Solve practice questions.
Day 5: Revise everything.


In [5]:
result = graph.invoke({
    "student_question": "Explain Artificial Intelligence",
    "task": "",
    "answer": ""
})
print(result["answer"])

Topic: Explain Artificial Intelligence

1. Understand the basic meaning of the topic.
2. Learn its important concepts.
3. Study simple examples.
4. Revise the key points.


In [6]:
result = graph.invoke({
    "student_question": "Give me a quiz on Python",
    "task": "",
    "answer": ""
})
print(result["answer"])

Sample Quiz

1. What is the main purpose of studying a topic?
2. Why is practice important?
3. How can revision improve learning?


In [7]:

!pip install -q -U langchain-google-genai


In [8]:

import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.3
)

print("Gemini Flash Lite model configured!")


Gemini Flash Lite model configured!


In [9]:

response = model.invoke("Reply with: Hello Mihika! Your AI Study Assistant is ready.")

print(response.content)


[{'type': 'text', 'text': 'Hello Mihika! Your AI Study Assistant is ready.', 'extras': {'signature': 'EnMKcQFpFH0TNkCT4i/yZegmrHTaen/VPV/Rf1joMNvWDpQT/ZoPtbGw/9ujtSBkRZLAcUMvpn2GKFFMVNmOC36xMjYfOKBzd25v8Pm8ozIVYbmzoF86PYzul3avz7a3Owo7IEe2Emj9KIH2PR/MhqbS/PJU'}}]


In [10]:

response = model.invoke(
    "Explain Artificial Intelligence in 3 simple sentences."
)

content = response.content

# Extract only the text from Gemini's response
if isinstance(content, list):
    answer = "\n".join(
        item["text"]
        for item in content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    answer = content

print(answer)


Artificial Intelligence is technology that allows computers to learn from information and perform tasks that typically require human intelligence. It works by analyzing large amounts of data to recognize patterns, make predictions, or solve complex problems. Essentially, it enables machines to mimic human thinking to help us work faster and more efficiently.


In [11]:

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Define the study assistant state
class StudyState(TypedDict):
    student_question: str
    task: str
    answer: str

# Decide which task the student wants
def classify_task(state: StudyState):
    question = state["student_question"].lower()

    if "quiz" in question or "question" in question:
        return {"task": "quiz"}
    elif "plan" in question or "schedule" in question:
        return {"task": "plan"}
    else:
        return {"task": "explain"}

# AI Topic Explanation
def explain_topic(state: StudyState):
    prompt = f"""
    You are a friendly AI tutor.
    Explain the following topic in simple English.
    Use a clear definition and an easy example.
    Student request: {state["student_question"]}
    """
    response = model.invoke(prompt)
    return {"answer": response.content}

# AI Study Planner
def make_plan(state: StudyState):
    prompt = f"""
    You are a study planner.
    Create a practical 5-day study plan in simple English.
    Include daily topics and practice.
    Student request: {state["student_question"]}
    """
    response = model.invoke(prompt)
    return {"answer": response.content}

# AI Quiz Generator
def make_quiz(state: StudyState):
    prompt = f"""
    You are a quiz generator.
    Create 5 relevant questions about the student's requested topic.
    Include answers at the end.
    Use simple English.
    Student request: {state["student_question"]}
    """
    response = model.invoke(prompt)
    return {"answer": response.content}

# Build the LangGraph workflow
builder = StateGraph(StudyState)

builder.add_node("classify", classify_task)
builder.add_node("explain", explain_topic)
builder.add_node("plan", make_plan)
builder.add_node("quiz", make_quiz)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    lambda state: state["task"],
    {
        "explain": "explain",
        "plan": "plan",
        "quiz": "quiz"
    }
)

builder.add_edge("explain", END)
builder.add_edge("plan", END)
builder.add_edge("quiz", END)

study_app = builder.compile()

print("AI Study Assistant is ready!")


AI Study Assistant is ready!


In [12]:
result = study_app.invoke({
    "student_question": "Explain machine learning",
    "task": "",
    "answer": ""
})
print(result["answer"])

[{'type': 'text', 'text': 'Hello! I’m happy to help you understand **machine learning**. Think of me as your friendly tutor!\n\n### What is Machine Learning?\n\nIn simple terms, **machine learning** is a way of teaching computers to learn from experience, just like humans do.\n\nNormally, if you want a computer to do something, you have to give it a long list of strict rules to follow. With machine learning, instead of giving the computer rules, you give it a **ton of data** (examples). The computer looks for patterns in that data and teaches itself how to make decisions or predictions.\n\n***\n\n### An Easy Example: The "Fruit Sorter"\n\nImagine you want a computer to tell the difference between an **apple** and a **banana**.\n\n*   **The Old Way (Programming):** You would have to write complex rules like: *"If the object is round and red, it is an apple. If it is long and yellow, it is a banana."* But what if you find a green apple? Or a brown banana? The computer would get confused 

In [13]:
result = study_app.invoke({
    "student_question": "Create a quiz about Artificial Intelligence",
    "task": "",
    "answer": ""
})
print(result["answer"])

[{'type': 'text', 'text': 'Here is a 5-question quiz about Artificial Intelligence:\n\n**1. What does the abbreviation "AI" stand for?**\nA) Automatic Internet\nB) Artificial Intelligence\nC) Advanced Information\n\n**2. Which of these is an example of an AI tool that can write text and answer questions?**\nA) ChatGPT\nB) A calculator\nC) A physical light switch\n\n**3. What do we call the process where AI "learns" from large amounts of data to improve its performance?**\nA) Machine Learning\nB) Human Reading\nC) Data Deleting\n\n**4. Which of these is a common use for AI in everyday life?**\nA) Recommending movies on Netflix\nB) Making a sandwich\nC) Cleaning the windows by hand\n\n**5. Can AI "think" and feel emotions exactly like a human being?**\nA) Yes, it has feelings.\nB) No, it uses math and patterns to simulate human-like responses.\nC) Yes, it is alive.\n\n***\n\n### Answer Key:\n1. **B** (Artificial Intelligence)\n2. **A** (ChatGPT)\n3. **A** (Machine Learning)\n4. **A** (Re

In [14]:

from langgraph.checkpoint.memory import InMemorySaver

# Create a memory checkpointer
memory = InMemorySaver()

# Recompile the existing graph with memory
study_app_memory = builder.compile(checkpointer=memory)

print("Conversation memory is ready!")


Conversation memory is ready!


In [15]:

from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "mihika_session_1"}}

# First message
result = study_app_memory.invoke(
    {
        "student_question": "My name is Mihika. I am learning Python.",
        "task": "explain",
        "answer": ""
    },
    config=config
)

print(result["answer"])


[{'type': 'text', 'text': 'Hi Mihika! It’s great to meet you. I’m happy to help you on your Python journey.\n\nToday, let’s talk about **Variables**.\n\n### What is a Variable?\nThink of a variable as a **labeled box**. In Python, you use these boxes to store information (like numbers or words) so that you can use them later in your code. You give the box a name, and then you put something inside it.\n\n### An Easy Example\nImagine you are writing a program for a game and you want to keep track of a player\'s score.\n\nIn Python, you would write it like this:\n\n```python\nscore = 10\n```\n\n**Here is what is happening:**\n*   **`score`**: This is the name of your box (the variable).\n*   **`=`**: This is the "assignment" operator. It tells Python, "Take the value on the right and put it into the box on the left."\n*   **`10`**: This is the information you are storing inside the box.\n\nNow, whenever you type `score` later in your code, Python will remember that it equals `10`. If the 

In [16]:

result = study_app_memory.invoke(
    {
        "student_question": "What is my name, and what am I learning?",
        "task": "explain",
        "answer": ""
    },
    config=config
)

print(result["answer"])


[{'type': 'text', 'text': 'Hello! I am your friendly AI tutor, and I would be happy to help you.\n\n### What is your name?\nSince I am an AI, I don\'t have access to your personal identity unless you have shared it with me in our conversation. If you haven\'t told me your name yet, I don\'t know it! **Would you like to tell me what I should call you?**\n\n### What are you learning?\nRight now, you are learning how to interact with an AI. You are testing my ability to explain concepts, answer questions, and provide helpful information.\n\n***\n\n### A Quick Lesson: What is an AI?\n\n**Definition:** \nAn **AI** (Artificial Intelligence) is a type of computer program that is designed to "think" or solve problems in a way that mimics human intelligence. It learns from large amounts of information so it can answer questions, write stories, or help with tasks.\n\n**An Easy Example:**\nThink of a **smart kitchen timer**. \n*   A regular timer just counts down until it hits zero. \n*   An **AI

In [18]:

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("Memory imports completed successfully!")


Memory imports completed successfully!


In [19]:

class MemoryStudyState(MessagesState):
    student_question: str
    answer: str

def study_with_memory(state: MemoryStudyState):
    system_message = SystemMessage(
        content=(
            "You are a helpful AI Study Assistant. "
            "Answer in simple English. "
            "Remember details shared earlier in this conversation "
            "and use them when answering follow-up questions."
        )
    )

    response = model.invoke(
        [system_message] + state["messages"]
    )

    content = response.content

    if isinstance(content, list):
        answer = "\n".join(
            item["text"]
            for item in content
            if isinstance(item, dict)
            and item.get("type") == "text"
        )
    else:
        answer = str(content)

    return {
        "answer": answer,
        "messages": [AIMessage(content=answer)]
    }

memory_builder = StateGraph(MemoryStudyState)

memory_builder.add_node("study", study_with_memory)
memory_builder.add_edge(START, "study")
memory_builder.add_edge("study", END)

memory = InMemorySaver()
study_memory_app = memory_builder.compile(checkpointer=memory)

print("Memory-enabled Study Assistant is ready!")


Memory-enabled Study Assistant is ready!


In [20]:

config = {
    "configurable": {
        "thread_id": "mihika_test_1"
    }
}

response1 = study_memory_app.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Mihika, and I am learning Python."
            )
        ],
        "student_question": "My name is Mihika, and I am learning Python."
    },
    config=config
)

print("AI:", response1["answer"])


AI: Hello, Mihika! It is nice to meet you. That is exciting that you are learning Python—it is a fantastic language to start with because it is very readable and powerful.

Since you are just starting out, how is it going so far? Are you working on any specific projects, or are you currently learning the basics like variables, loops, or functions? 

I am here to help whenever you get stuck or have a question!


In [21]:

response2 = study_memory_app.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is my name, and what am I learning?"
            )
        ],
        "student_question": "What is my name, and what am I learning?"
    },
    config=config
)

print("AI:", response2["answer"])


AI: Your name is **Mihika**, and you are learning **Python**.


In [22]:

!pip install gradio


In [23]:

import gradio as gr
from langchain_core.messages import HumanMessage

def chat_with_study_assistant(message, history):
    config = {
        "configurable": {
            "thread_id": "mihika_gradio_session"
        }
    }

    result = study_memory_app.invoke(
        {
            "messages": [HumanMessage(content=message)],
            "student_question": message
        },
        config=config
    )

    return result["answer"]


demo = gr.ChatInterface(
    fn=chat_with_study_assistant,
    title="AI Study Assistant",
    description="Ask questions, learn topics, and explore your studies with AI.",
    examples=[
        "Explain Artificial Intelligence in simple English.",
        "Make a 7-day Python study plan.",
        "Create a quiz about machine learning."
    ]
)

print("Gradio interface is ready!")


Gradio interface is ready!


In [24]:

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://41ee45b6c91e99d7d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://41ee45b6c91e99d7d2.gradio.live


In [26]:

response = model.invoke(
    "Explain Artificial Intelligence in 2 simple sentences."
)

content = response.content

if isinstance(content, list):
    answer = "\n".join(
        item["text"]
        for item in content
        if isinstance(item, dict) and item.get("type") == "text"
    )
else:
    answer = str(content)

print(answer)


Artificial intelligence is the ability of a computer program to perform tasks that typically require human intelligence, such as learning, reasoning, and problem-solving. It works by analyzing vast amounts of data to identify patterns and make decisions or predictions based on what it has learned.


In [31]:

import gradio as gr
from langchain_core.messages import HumanMessage

def chat_with_study_assistant(message, history, request: gr.Request):
    config = {
        "configurable": {
            "thread_id": request.session_hash
        }
    }

    try:
        result = study_memory_app.invoke(
            {
                "messages": [HumanMessage(content=message)],
                "student_question": message
            },
            config=config
        )

        return result["answer"]

    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            return (
                "Gemini's free API quota has been reached. "
                "Please try again after the quota resets. "
                "Your project code is still available."
            )

        print("Chatbot error:", e)
        return "Something went wrong. Please check the Colab output."


demo = gr.ChatInterface(
    fn=chat_with_study_assistant,
    title="AI Study Assistant",
    description="Ask questions, learn topics, and explore your studies with AI.",
    examples=[
        "Explain Artificial Intelligence in simple English.",
        "Make a 7-day Python study plan.",
        "Create a quiz about machine learning."
    ]
)

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c961d7605e819fd89a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
